# Dataset Registry

## Business objective
Persist shared dataset definitions in S3 as Parquet so that Databricks
workflows can discover enabled sources without relying on notebook memory.

Run this notebook only when a dataset definition is added or changed.

In [0]:
%run ../00_project_setup/00_project_setup

# AI Workforce Capacity Planning Platform

## Project Setup

This notebook centralizes the shared S3 paths and configuration used by
all Databricks notebooks in the project.

### Storage architecture

- Landing: original source files
- Bronze: standardized Parquet datasets
- Silver: cleaned and validated Parquet datasets
- Gold: model-ready and business-ready Parquet datasets
- Databricks: processing and orchestration
- Amazon S3: persistent storage

This notebook is imported by other notebooks and is not scheduled as an
independent workflow task.

## Lightweight storage validation

The setup notebook checks that the project root is listable.  
It deliberately avoids writing test data every time another notebook imports it.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# ======================================================
# Dataset Registry Schema
# ======================================================

registry_schema = StructType(
    [
        StructField("dataset_id", IntegerType(), False),
        StructField("dataset_name", StringType(), False),
        StructField("dataset_key", StringType(), False),
        StructField("dataset_version", StringType(), False),
        StructField("dataset_owner", StringType(), False),
        StructField("source_type", StringType(), False),

        # Human-readable dataset webpage
        StructField("source_location", StringType(), False),

        # Machine-readable provider identifier
        StructField("source_reference", StringType(), False),

        StructField("source_format", StringType(), False),
        StructField("landing_folder", StringType(), False),
        StructField("enabled", BooleanType(), False),
        StructField("status", StringType(), False),
    ]
)

dataset_registry = [
    (
        1,
        "DataCo SMART Supply Chain",
        "dataco_supply_chain",
        "1.0",
        "Issouf KABRE",

        "Kaggle",

        # Human readable page
        "https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis",

        # Machine readable identifier
        "shashwatwork/dataco-smart-supply-chain-for-big-data-analysis",

        "csv",

        "dataco_supply_chain",

        True,

        "READY_FOR_DOWNLOAD",
    )
]

registry_df = spark.createDataFrame(
    dataset_registry,
    schema=registry_schema,
)

# Reject duplicate business keys before persistence.
duplicate_keys = (
    registry_df
    .groupBy("dataset_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_keys > 0:
    raise ValueError(
        "Dataset registry contains duplicate dataset_key values."
    )

# Validate required source URLs.
invalid_source_locations = (
    registry_df
    .filter(
        F.col("source_location").isNull()
        | (F.trim(F.col("source_location")) == "")
        | F.col("source_location").contains("<owner>")
        | F.col("source_location").contains("<dataset-name>")
    )
    .count()
)

if invalid_source_locations > 0:
    print(
        f"{invalid_source_locations} dataset(s) still require "
        "a verified source URL."
    )

display(registry_df)

dataset_id,dataset_name,dataset_key,dataset_version,dataset_owner,source_type,source_location,source_reference,source_format,landing_folder,enabled,status
1,DataCo SMART Supply Chain,dataco_supply_chain,1.0,Issouf KABRE,Kaggle,https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis,shashwatwork/dataco-smart-supply-chain-for-big-data-analysis,csv,dataco_supply_chain,true,READY_FOR_DOWNLOAD


In [0]:
(
    registry_df.write
    .mode("overwrite")
    .parquet(DATASET_REGISTRY_PATH)
)

print(f"Dataset registry written to: {DATASET_REGISTRY_PATH}")

Dataset registry written to: s3a://issouf-data-lake/overtime-capacity-planning/registry/datasets


In [0]:
persisted_registry_df = spark.read.parquet(
    DATASET_REGISTRY_PATH
)

display(persisted_registry_df)
persisted_registry_df.printSchema()

dataset_id,dataset_name,dataset_key,dataset_version,dataset_owner,source_type,source_location,source_reference,source_format,landing_folder,enabled,status
1,DataCo SMART Supply Chain,dataco_supply_chain,1.0,Issouf KABRE,Kaggle,https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis,shashwatwork/dataco-smart-supply-chain-for-big-data-analysis,csv,dataco_supply_chain,true,READY_FOR_DOWNLOAD


root
 |-- dataset_id: integer (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- dataset_key: string (nullable = true)
 |-- dataset_version: string (nullable = true)
 |-- dataset_owner: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- source_location: string (nullable = true)
 |-- source_reference: string (nullable = true)
 |-- source_format: string (nullable = true)
 |-- landing_folder: string (nullable = true)
 |-- enabled: boolean (nullable = true)
 |-- status: string (nullable = true)

